# LoRA Background Generation Integration

This notebook demonstrates the integration of a LoRA (Low-Rank Adaptation) model with a background generation model.
It processes images by replacing their backgrounds using two methods:
1. The original ControlNet-BG-Gen model without LoRA
2. The ControlNet-BG-Gen model with a trained LoRA model 

The notebook showcases how the LoRA model influences the background generation process,
adapting it to the style learned from the training dataset.

## Setup and Dependencies

Let's start by importing all the necessary libraries and defining some utility functions.

In [ ]:
import os
import torch
import gc
from diffusers import AutoencoderKL, EulerAncestralDiscreteScheduler, LCMScheduler
from replace_bg.model.pipeline_controlnet_sd_xl import StableDiffusionXLControlNetPipeline
from replace_bg.model.controlnet import ControlNetModel
from replace_bg.utilities import resize_image, remove_bg_from_image, paste_fg_over_image, get_control_image_tensor
from PIL import Image

## Utility Functions

We'll define several utility functions to help with processing images and managing GPU memory.

In [ ]:
# Function to clear GPU memory
def clear_gpu_memory():
    torch.cuda.empty_cache()
    gc.collect()

# Function to process a single image
def process_image(image_path, prompt, negative_prompt, pipe, num_inference_steps=50):
    # Load and preprocess the image
    image = Image.open(image_path)
    image = resize_image(image)
    mask = remove_bg_from_image(image_path)
    control_tensor = get_control_image_tensor(pipe.vae, image, mask)

    # Generate new background
    seed = 0
    generator = torch.Generator("cuda").manual_seed(seed)
    gen_img = pipe(
        negative_prompt=negative_prompt,
        prompt=prompt,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=num_inference_steps,
        image=control_tensor,
        generator=generator
    ).images[0]

    # Combine new background with original foreground
    result_image = paste_fg_over_image(gen_img, image, mask)
    return result_image

# Function to process all images in a directory
def process_all_images(pipe, input_dir, output_dir, prompt, negative_prompt, prefix, num_inference_steps=50):
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_path = os.path.join(input_dir, filename)
            result_image = process_image(input_path, prompt, negative_prompt, pipe, num_inference_steps)
            output_path = os.path.join(output_dir, f"{prefix}_{filename}")
            result_image.save(output_path)
            print(f"Saved {output_path}")

## Configuration

Set up the directories, prompts, and model configurations.

In [ ]:
# Define directories
input_dir = "./input_images"  # Directory containing input images
output_dir = "./output_images"  # Directory for saving results
os.makedirs(output_dir, exist_ok=True)

# Define prompt and negative prompt
# You can customize these prompts based on the background style you want
prompt = "A scenic background with natural elements, good lighting and depth. Suitable for a profile picture."
negative_prompt = "distracting elements, cluttered, busy, noisy, poor quality"

## Load Shared Components

Load the shared components that will be used by both pipeline configurations.

In [ ]:
# Load shared components
# Replace these paths with your model paths
controlnet = ControlNetModel.from_pretrained("path/to/your/controlnet/model", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("path/to/your/vae/model", torch_dtype=torch.float16)

## Process with Original Model

First, we'll process the images using the original model without any LoRA adaptation.

In [ ]:
# Process with original model
clear_gpu_memory()
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "path/to/your/base/model", 
    controlnet=controlnet, 
    torch_dtype=torch.float16, 
    vae=vae
).to('cuda:0')

pipe.scheduler = EulerAncestralDiscreteScheduler(
    beta_start=0.00085, 
    beta_end=0.012, 
    beta_schedule="scaled_linear",
    num_train_timesteps=1000, 
    steps_offset=1
)

process_all_images(pipe, input_dir, output_dir, prompt, negative_prompt, "original")
del pipe
clear_gpu_memory()

## Process with LoRA Model

Now, we'll process the same images using the base model with the LoRA weights trained on your custom style.

In [ ]:
# Process with LoRA model
clear_gpu_memory()
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "path/to/your/base/model", 
    controlnet=controlnet, 
    torch_dtype=torch.float16, 
    vae=vae
).to('cuda:0')

pipe.scheduler = EulerAncestralDiscreteScheduler(
    beta_start=0.00085, 
    beta_end=0.012, 
    beta_schedule="scaled_linear",
    num_train_timesteps=1000, 
    steps_offset=1
)

# Replace with path to your LoRA weights
lora_model_path = "path/to/your/lora/weights.safetensors"
pipe.load_lora_weights(lora_model_path)

# You can customize this prefix to describe your LoRA style
style_prefix = "custom_style"
process_all_images(pipe, input_dir, output_dir, prompt, negative_prompt, style_prefix)
del pipe
clear_gpu_memory()

## Conclusion

All images have been processed using both the original model and the LoRA-adapted model. The results are saved in the output directory with appropriate prefixes for comparison.

This demonstrates how LoRA can be used to adapt a pre-trained model to generate backgrounds in a specific style without requiring full model retraining.

In [ ]:
print("All images processed.")